In [1]:
import pandas as pd
import numpy as np
from abiasales.data_generation import *
from abiasales.cleaning import remove_outliers
from abiasales.power import calc_power_table
from abiasales.results import calc_exp_results
from abiasales.simulation import construct_exp_configs, run_aa_test_simulation, run_ab_test_simulation

In [2]:
import warnings
warnings.filterwarnings("ignore")
pd.options.display.float_format = '{:,.2f}'.format

# DATA

In [3]:
df_a = create_test_data(uplift=0, n_users=1000000)
df_a['exp_group'] = 'A'
df_b = create_test_data(uplift=0.1, n_users=1000000)
df_b['exp_group'] = 'B'

df = pd.concat([df_a, df_b]).reset_index(drop=True)
del df_a, df_b

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Data columns (total 12 columns):
 #   Column         Dtype  
---  ------         -----  
 0   uid            int64  
 1   platform       object 
 2   country        object 
 3   strata         object 
 4   clicks         float64
 5   payer_flag     int64  
 6   purchases      int64  
 7   views          float64
 8   views_cov      float64
 9   clicks_cov     float64
 10  purchases_cov  float64
 11  exp_group      object 
dtypes: float64(5), int64(3), object(4)
memory usage: 183.1+ MB


In [5]:
STRATS_COLS = ['platform', 'country']
FEATURE_COLS = ['views_cov', 'clicks_cov', 'purchases_cov']

In [6]:
df.groupby(['exp_group']).agg(
    users=('uid', 'count'),
    views=('views', 'mean'),
    clicks=('clicks', 'mean'),
    payers=('payer_flag', 'mean'),
    purchases=('purchases', 'mean'),
    views_cov=('views_cov', 'mean'),
    clicks_cov=('clicks_cov', 'mean'),
    purchases_cov=('purchases_cov', 'mean')
)

,users,views,clicks,payers,purchases,views_cov,clicks_cov,purchases_cov
exp_group,,,,,,,,
A,1000000,13.03,1.71,0.25,2.08,13.00,1.71,2.07
B,1000000,12.97,1.88,0.25,2.27,12.97,1.88,2.26


Check correlation of metric and covariates

In [7]:
from scipy.stats import pearsonr

print('Correlation')
for f in FEATURE_COLS:
    print(pearsonr(df['purchases'], df[f])[0])

Correlation
0.003351260005414922
0.013163818081884177
0.2987743771277356


Removing outliers

In [8]:
df = remove_outliers(df, metrics=FEATURE_COLS, contamination=0.001)

# SAMPLE SIZE

In [9]:
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    alpha=[0.05], power=[0.8], uplift=[0.01, 0.03, 0.05], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
1.0%,761253
3.0%,84584
5.0%,30451


In [10]:
# WITH STRATIFICATION
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    use_stratification=True,
    strats_cols=STRATS_COLS,
    alpha=[0.05], power=[0.8], uplift=[0.05], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
5.0%,26811


In [11]:
# WITH CUPED
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    var_reduction_method='cuped',
    var_reduction_covariates=FEATURE_COLS,
    alpha=[0.05], power=[0.8], uplift=[0.05], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
5.0%,27751


In [12]:
# WITH STRATIFICATION AND CUPED
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    use_stratification=True,
    strats_cols=STRATS_COLS,
    var_reduction_method='cuped',
    var_reduction_covariates=FEATURE_COLS,
    alpha=[0.05], power=[0.8], uplift=[0.05], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
5.0%,23163


# MDE

MDE calculation accounting for given sample size

In [13]:
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='mde',
    alpha=[0.05], power=[0.8], sample_size=[int(len(df) / 2)], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
999000,0.9%


In [14]:
# WITH STRATIFICATION AND CUPED
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='mde',
    use_stratification=True,
    strats_cols=STRATS_COLS,
    var_reduction_method='cuped',
    var_reduction_covariates=FEATURE_COLS,
    alpha=[0.05], power=[0.8], sample_size=[int(len(df) / 2)], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
999000,0.8%


# RESULTS

In [37]:
df_a = create_test_data(uplift=0, n_users=20000)
df_a['exp_group'] = 'A'
df_b = create_test_data(uplift=0.05, n_users=20000)
df_b['exp_group'] = 'B'

df_results = pd.concat([df_a, df_b]).reset_index(drop=True)
del df_a, df_b

Calculating experiment results

List of metric config parameters:
- **num** - name of the column containing the numerator of the metric.
- **den** - name of the column containing the denominator of the metric. Defaults to None and can be omitted.
- **weight_method** - type of user weighting. Default is 'uniform'. Possible values: 'uniform', 'size', 'sqrt', 'intra_corr'.
- **stat_method** - statistical test used to calculate the p-value. Default is 't_test'. Possible values: 'z_test', 't_test', 'proportion' (for conversion metrics only), 'bootstrap'.
- **use_delta_method** - whether to use the delta method for variance estimation (relevant only for ratio metrics).
- **var_reduction_method** - variance reduction method. Default is None. Possible values: None, 'cuped', 'cupac'.
- **var_reduction_covariates** - list of covariates for CUPED/CUPAC. Defaults to None. Ignored if var_reduction_method = None.
- **handle_outliers** - how to handle outliers. Default is None. Posible values: remove (IsolationTree), cap (capping values).
- **outliers_contamination** - threshold to define an outlier. Default is 0.001. In case of handle_outliers="remove" – top outliers_contamination % of samples, in case of handle_outliers="cap" (capping values) – top quantile level of capping.
- **outliers_cols** – columns to detect outliers, relevant only for handle_outliers="remove".

Stratification is configured at the calc_exp_results function level:
- **use_stratification** - whether to apply stratification. Defaults to False.
- **strats_cols** - list of columns used for stratification. Defaults to None; ignored if use_stratification = False.

The function returns two p-value values:
- **pvalue_init** – the initial p-value obtained from the statistical test.
- **pvalue** – the p-value adjusted for multiple hypothesis testing.

In [31]:
# Target metrics
metrics = {
    'purchases': {
        'num': 'purchases',
        'weight_method': 'uniform',
        'stat_method': 't_test'
    },
    'CTR': {
        'num': 'clicks',
        'den': 'views',
        'weight_method': 'size',
        'stat_method': 't_test'
    }
}

# Same metrics but with CUPED and outliers handling
metrics_cuped = {
    'purchases': {
        'num': 'purchases',
        'weight_method': 'uniform',
        'stat_method': 't_test',
        'var_reduction_method': 'cuped',
        'var_reduction_covariates': FEATURE_COLS,
        'handle_outliers': 'remove',
        'outliers_cols': FEATURE_COLS,
        'outliers_contamination': 0.001
    },
    'CTR': {
        'num': 'clicks',
        'den': 'views',
        'weight_method': 'size',
        'stat_method': 't_test',
        'var_reduction_method': 'cuped',
        'var_reduction_covariates': FEATURE_COLS,
        'handle_outliers': 'remove',
        'outliers_cols': FEATURE_COLS,
        'outliers_contamination': 0.001
    }
}

In [38]:
results = calc_exp_results(
    df_results, exp_group_col='exp_group', metrics=metrics
)

,metric,group_1,group_2,mean_control,mean_treatment,uplift,confint,pvalue_init,pvalue
0,purchases,A,B,2.060700,2.120500,2.9%,"('-1.59%', '7.42%')",0.200332,0.200332
1,CTR,A,B,0.133730,0.138499,3.57%,"('2.08%', '5.05%')",0.000002,0.000003


In [39]:
# WITH STRATIFICATION AND CUPED
results = calc_exp_results(
    df_results, exp_group_col='exp_group', metrics=metrics_cuped,
    use_stratification=True, strats_cols=STRATS_COLS
)

,metric,group_1,group_2,mean_control,mean_treatment,uplift,confint,pvalue_init,pvalue
0,purchases,A,B,2.056773,2.119597,3.05%,"('-0.9%', '7.02%')",0.124895,0.234190
1,CTR,A,B,0.132019,0.135413,2.57%,"('-1.29%', '6.45%')",0.187558,0.234190


# SIMULATION

In this section, we will simulate A/A and A/B tests to verify that the p-value properly controls the Type I error rate and to compare the power of different statistical tests.

Important note: There are multiple ways to generate synthetic uplift for A/B test simulations, and the results can vary significantly depending on the chosen approach.
The library implements the following options:

- **proportion** – the simplest option, where uplift can be generated directly (suitable for proportion-type metrics).
- **regular** – used for metrics such as profit, purchases, etc. A normal random variable with a mean equal to the absolute uplift value is added. This is a generally applicable and versatile option.
- **ratio** – the most complex case. The library adds a normal random component to all units where the metric value is not zero. However, this approach has certain drawbacks — for instance, it interacts poorly with CUPED.

Each of the functions **`run_aa_test_simulation`** and **`run_ab_test_simulation`** takes a **`configs`** argument — a list of dictionaries, where each dictionary defines the metric configuration and the statistical methods to be used.  

Each dictionary has the following **structure**:

---

- **`metric_num`** *(str)*  
  Name of the column with the metric’s numerator (e.g., `"purchases"`).

- **`metric_den`** *(str)*  
  Name of the column with the metric’s denominator.  
  If `None`, assumes a value of 1.

- **`weight_method`** *(str)*  
  Unit-level weighting scheme to apply.  
  Allowed:  
  - `'uniform'`: all users weight = 1  
  - `'size'`: weights equal to the metric “size” (e.g., denominator or exposure)  
  - `'sqrt'`: weights are √(size)  
  - `'intra_corr'`: correlation-aware weights (accounts for within-user correlation)  
  Default: `'uniform'`.

- **`stat_method`** *(str)*  
  Statistical test to compute the p-value.  
  Allowed: `'proportion'`, `'z_test'`, `'t_test'`, `'bootstrap'`.  
  Use `'proportion'` only for conversion metrics.  
  Default: `'t_test'`.

- **`use_stratification`** *(bool)*  
  Whether to apply stratification.  
  If `False`, `strats_cols` are ignored.  
  Default: `False`.

- **`strats_cols`** *(list[str] | None)*  
  Columns to use as strata keys (e.g., `device`, `country`).  
  Ignored if `use_stratification=False`.  
  Default: `None`.

- **`var_reduction_method`** *(str)*  
  Variance reduction method.  
  Allowed: `None`, `'cuped'`, `'cupac'`.  
  Default: `None`.

- **`var_reduction_covariates`** *(list[str] | None)*  
  Covariates for CUPED/CUPAC.  
  Ignored if `var_reduction_method=None`.  
  Default: `None`.

- **`handle_outliers`** *(str | None)*  
  Strategy for outliers.  
  Allowed:  
  - `None`  
  - `'remove'`: drop outliers using IsolationTree  
  - `'cap'`: cap metric at quantile threshold  
  Default: `None`.

- **`outliers_contamination`** *(float | None)*  
  Expected outlier fraction (e.g., `0.001` = 0.1%).  
  Used to set cutoffs for `'remove'`/`'cap'`.  
  Ignored if `handle_outliers=None`.  
  Default: `0.01`.

- **`outliers_cols`** *(list[str])*  
  Columns on which to detect/handle outliers (e.g., `exposure`, `spend`, `price`).  
  Ignored if `handle_outliers != 'remove'`.

---

**Example:**

```python
{
    'metric_num': 'purchases',
    'weight_method': 'uniform',
    'stat_method': 'z_test',
    'use_stratification': False,
    'strats_cols': STRATS_COLS,                       # ignored because use_stratification=False
    'var_reduction_method': ['country'],
    'var_reduction_covariates': ['purchases_cov'],    # ignored because var_reduction_method=None
    'handle_outliers': None,
    'outliers_contamination': 0.001,                  # ignored because handle_outliers=None
    'outliers_cols': ['purchases_cov']                # ignored because handle_outliers=None
}


In [40]:
df_sim = df[df['exp_group']=='A'] # control group sample
n_exp = 1000 # number of experiments in simulations

## PROPORTION

In [133]:
sample_size = 1000

pt, pw = calc_power_table(
    df=df_sim,
    metric_num='payer_flag',
    mode='mde',
    alpha=[0.05], power=[0.8], sample_size=[sample_size], control_perc=0.5,
    return_power_list=True
)
display(pt)

model_uplift = pw[0]

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
1000,21.7%


Compare different statistical tests

In [134]:
configs = construct_exp_configs(
    metric_num='payer_flag',
    weight_methods=['uniform'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    stat_methods=['proportion', 'z_test', 't_test'], # 'proportion', 'z_test', 't_test', 'bootstrap'
    use_stratification=[False], # False, True
    strats_cols=STRATS_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

First, run synthetic A/A tests.

In [135]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'payer_flag', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'proportion', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:14<00:00, 69.50it/s]


Running A/A for config: {'num': 'payer_flag', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'z_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:14<00:00, 70.91it/s]


Running A/A for config: {'num': 'payer_flag', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:14<00:00, 70.72it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,uniform,False,False,proportion,False,None,None,0.001000,0.494856,True
1,uniform,False,False,z_test,False,None,None,0.001000,0.493643,True
2,uniform,False,False,t_test,False,None,None,0.001000,0.485237,True


All tests are valid.

Second, run A/B-тесты in order to compare powers of tests

In [136]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='proportion',
    uplift=model_uplift
)

Running A/B for config: {'num': 'payer_flag', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'proportion', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [03:10<00:00,  5.25it/s]


Running A/B for config: {'num': 'payer_flag', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'z_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [03:11<00:00,  5.23it/s]


Running A/B for config: {'num': 'payer_flag', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [03:11<00:00,  5.22it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,power
0,uniform,False,False,proportion,False,None,None,0.001000,0.795000
1,uniform,False,False,z_test,False,None,None,0.001000,0.798000
2,uniform,False,False,t_test,False,None,None,0.001000,0.801000


An illustrative example – all statistical tests show the same power. For conversion metrics, even with relatively small sample sizes (e.g., 1000 in this example), there is no practical difference in which test is used.

## VALUE

In [46]:
sample_size = 1000

pt, pw = calc_power_table(
    df=df_sim,
    metric_num='purchases',
    mode='mde',
    alpha=[0.05], power=[0.8], sample_size=[sample_size], control_perc=0.5,
    return_power_list=True
)
display(pt)

model_uplift = pw[0]

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
1000,27.9%


First, compare 3 main statistical tests

In [138]:
configs = construct_exp_configs(
    metric_num='purchases',
    weight_methods=['uniform'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    stat_methods=['z_test', 't_test', 'bootstrap'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[False], # False, True
    strats_cols=STRATS_COLS,
    var_reduction_methods=[None], # None, 'cuped'
    var_reduction_covariates=FEATURE_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [139]:
configs

[{'num': 'purchases',
  'den': None,
  'weight_method': 'uniform',
  'apply_linearization': False,
  'use_delta_method': False,
  'stat_method': 'z_test',
  'use_stratification': False,
  'strats_cols': ['platform', 'country'],
  'var_reduction_method': None,
  'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'],
  'handle_outliers': None,
  'outliers_contamination': 0.001,
  'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov']},
 {'num': 'purchases',
  'den': None,
  'weight_method': 'uniform',
  'apply_linearization': False,
  'use_delta_method': False,
  'stat_method': 't_test',
  'use_stratification': False,
  'strats_cols': ['platform', 'country'],
  'var_reduction_method': None,
  'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'],
  'handle_outliers': None,
  'outliers_contamination': 0.001,
  'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov']},
 {'num': 'purchases',
  'den': None,
  'weight_method': 'uniform',
  'ap

In [48]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'z_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:14<00:00, 68.01it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:14<00:00, 68.51it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'bootstrap', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [02:30<00:00,  6.64it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,uniform,False,False,z_test,False,None,None,0.001000,0.502959,True
1,uniform,False,False,t_test,False,None,None,0.001000,0.477310,True
2,uniform,False,False,bootstrap,False,None,None,0.001000,0.485020,True


All the tests are valid.

Compare powers:

In [49]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='value',
    uplift=model_uplift
)

Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'z_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:19<00:00, 50.24it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:20<00:00, 49.74it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'bootstrap', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [02:31<00:00,  6.60it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,power
0,uniform,False,False,z_test,False,None,None,0.001000,0.798000
1,uniform,False,False,t_test,False,None,None,0.001000,0.792000
2,uniform,False,False,bootstrap,False,None,None,0.001000,0.654971


Another illustrative case: there is no difference between the z-test and the t-test, since asymptotically they are equivalent. In practice, sample sizes are usually large (10k in this example), so the choice of statistical test makes no real difference.<br>
It is also noteworthy that the bootstrap shows lower power. This can happen when the underlying distributions are highly skewed.

Check how stratification works

In [50]:
configs = construct_exp_configs(
    metric_num='purchases',
    weight_methods=['uniform'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[False, True], # False, True
    strats_cols=STRATS_COLS,
    var_reduction_methods=[None], # None, 'cuped'
    var_reduction_covariates=FEATURE_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [51]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:14<00:00, 66.95it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:19<00:00, 50.52it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,uniform,False,False,t_test,False,None,None,0.001000,0.501737,True
1,uniform,False,False,t_test,True,None,None,0.001000,0.508054,True


In [52]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='value',
    uplift=model_uplift
)

Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:19<00:00, 50.15it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:24<00:00, 40.19it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,power
0,uniform,False,False,t_test,False,None,None,0.001000,0.766000
1,uniform,False,False,t_test,True,None,None,0.001000,0.821000


We observe roughly a 0.05 increase in power due to stratification.<br>
This example is based on synthetic data, which is why the gain is relatively large.
In practice, the improvement is likely to be smaller.
However, adding stratification is almost always a cost-free operation, so there’s little reason to ignore it.

Try to add CUPED and CUPAC

In [53]:
configs = construct_exp_configs(
    metric_num='purchases',
    weight_methods=['uniform'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[True], # False, True
    strats_cols=STRATS_COLS,
    var_reduction_methods=[None, 'cuped', 'cupac'], # None, 'cuped'
    var_reduction_covariates=FEATURE_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [54]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:20<00:00, 49.02it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:54<00:00, 18.28it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cupac', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [04:44<00:00,  3.51it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,uniform,False,False,t_test,True,None,None,0.001000,0.490302,True
1,uniform,False,False,t_test,True,cuped,None,0.001000,0.506158,True
2,uniform,False,False,t_test,True,cupac,None,0.001000,0.599240,False


CUPAC is not valid here.

In [55]:
configs = [s for s in configs if not (s['var_reduction_method']=='cupac')]

Compare powers

In [56]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='value',
    uplift=model_uplift
)

Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:25<00:00, 39.59it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:58<00:00, 16.98it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,power
0,uniform,False,False,t_test,True,None,None,0.001000,0.824000
1,uniform,False,False,t_test,True,cuped,None,0.001000,0.889000


CUPED provides a clear improvement in statistical power.

Compare handling outliers strategies

In [60]:
configs = construct_exp_configs(
    metric_num='purchases',
    weight_methods=['uniform'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[True], # False, True
    strats_cols=STRATS_COLS,
    var_reduction_methods=['cuped'], # None, 'cuped'
    var_reduction_covariates=FEATURE_COLS,
    handle_outliers=[None, 'remove', 'cap'], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [61]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:53<00:00, 18.87it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': 'remove', 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [10:34<00:00,  1.58it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': 'cap', 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:56<00:00, 17.62it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,uniform,False,False,t_test,True,cuped,None,0.001000,0.511004,True
1,uniform,False,False,t_test,True,cuped,remove,0.001000,0.501881,True
2,uniform,False,False,t_test,True,cuped,cap,0.001000,0.504131,True


In [62]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='value',
    uplift=model_uplift
)

Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:58<00:00, 17.12it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': 'remove', 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [11:19<00:00,  1.47it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': 'cap', 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:01<00:00, 16.17it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,power
0,uniform,False,False,t_test,True,cuped,None,0.001000,0.874000
1,uniform,False,False,t_test,True,cuped,remove,0.001000,0.957000
2,uniform,False,False,t_test,True,cuped,cap,0.001000,0.877000


Removing outliers by pre-experiment metrics looks much better than other options

Compare outliers contamination levels

In [64]:
configs = construct_exp_configs(
    metric_num='purchases',
    weight_methods=['uniform'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[True], # False, True
    strats_cols=STRATS_COLS,
    var_reduction_methods=['cuped'], # None, 'cuped'
    var_reduction_covariates=FEATURE_COLS,
    handle_outliers=['remove'], # None, 'remove', 'cap',
    outliers_contamination=[0.001, 0.005, 0.01], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [65]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': 'remove', 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [10:36<00:00,  1.57it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': 'remove', 'outliers_contamination': 0.005, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [10:38<00:00,  1.57it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': 'remove', 'outliers_contamination': 0.01, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [10:36<00:00,  1.57it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,uniform,False,False,t_test,True,cuped,remove,0.001000,0.497468,True
1,uniform,False,False,t_test,True,cuped,remove,0.005000,0.511098,True
2,uniform,False,False,t_test,True,cuped,remove,0.010000,0.508548,True


In [66]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='value',
    uplift=model_uplift
)

Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': 'remove', 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [11:18<00:00,  1.47it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': 'remove', 'outliers_contamination': 0.005, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [11:17<00:00,  1.48it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': 'remove', 'outliers_contamination': 0.01, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [11:19<00:00,  1.47it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,power
0,uniform,False,False,t_test,True,cuped,remove,0.001000,0.955000
1,uniform,False,False,t_test,True,cuped,remove,0.005000,0.962000
2,uniform,False,False,t_test,True,cuped,remove,0.010000,0.950000


Treating 0.5% of samples as outliers and removing them gives a little bit better statistical power.

Overall, best setup is:
- t-test
- Use stratification
- Use CUPED
- Remove 0.5% of outliers by pre-experiment features

It gives 15% increase in statistical power comparing to default t-test without anything.

## RATIO

In [105]:
sample_size = 10000

pt, pw = calc_power_table(
    df=df_sim,
    metric_num='clicks',
    metric_den='views',
    weight_method='size',
    use_delta_method=True,
    mode='mde',
    alpha=[0.05], power=[0.8], sample_size=[sample_size], control_perc=0.5,
    return_power_list=True
)
display(pt)

model_uplift = pw[0]

ERROR! Session/line number was not unique in database. History logging moved to new session 1401


"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
10000,2.9%


Compare two weighting methods, the use of linearization, and the delta method.

In [121]:
configs = construct_exp_configs(
    metric_num='clicks',
    metric_den='views',
    weight_methods=['uniform', 'size'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    apply_linearization=[False, True], # False, True
    use_delta_method=[False, True], # False, True
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[False], # False, True
    strats_cols=STRATS_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

Filter out obviously uninteresting combinations

In [122]:
configs = [
    s for s in configs
    if
        not (s['weight_method']=='size' and s['apply_linearization'])
        and not (s['apply_linearization'] and s['weight_method']!='uniform')
        and not (not s['apply_linearization'] and s['weight_method']=='uniform')
]

Simulate A/A-tests

In [108]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'uniform', 'apply_linearization': True, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:36<00:00, 10.35it/s]


Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:36<00:00, 10.40it/s]


Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': True, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:36<00:00, 10.34it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,uniform,True,False,t_test,False,None,None,0.001000,0.507237,True
1,size,False,False,t_test,False,None,None,0.001000,0.546580,False
2,size,False,True,t_test,False,None,None,0.001000,0.498475,True


The default ratio metric calculation (with size-based weighting) starts to show a deviation in Type I error.

In [123]:
configs = [s for s in configs if not ((s['weight_method']=='size') and (s['use_delta_method']==False))]

Compare powers:

In [125]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='ratio',
    uplift=model_uplift
)

Running A/B for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'uniform', 'apply_linearization': True, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:49<00:00,  9.17it/s]


Running A/B for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': True, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:48<00:00,  9.22it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,power
0,uniform,True,False,t_test,False,None,None,0.001000,0.681000
1,size,False,True,t_test,False,None,None,0.001000,0.680000


All options perform roughly the same, so we’ll keep only size + delta_method as the default. Here 80% of power is not achieved because it's hard to accurately inject an effect to ratio metric in simulated A/B-tests.

Now let’s try adding stratification.

In [126]:
configs = construct_exp_configs(
    metric_num='clicks',
    metric_den='views',
    weight_methods=['size'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    apply_linearization=[False], # False, True
    use_delta_method=[True], # False, True
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[False, True], # False, True
    strats_cols=STRATS_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [127]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': True, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:42<00:00,  9.75it/s]


Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': True, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [02:12<00:00,  7.53it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,size,False,True,t_test,False,None,None,0.001000,0.496743,True
1,size,False,True,t_test,True,None,None,0.001000,0.499453,True


In [128]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='ratio',
    uplift=model_uplift
)

Running A/B for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': True, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:46<00:00,  9.35it/s]


Running A/B for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': True, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [02:17<00:00,  7.27it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,power
0,size,False,True,t_test,False,None,None,0.001000,0.680000
1,size,False,True,t_test,True,None,None,0.001000,0.755000


Stratification provides a substantial increase in statistical power.

Now let’s compare different weighting methods.

In [129]:
configs = construct_exp_configs(
    metric_num='clicks',
    metric_den='views',
    weight_methods=['size', 'sqrt', 'intra_corr'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    apply_linearization=[False], # False, True
    use_delta_method=[False], # False, True
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[True], # False, True
    strats_cols=STRATS_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [130]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [02:09<00:00,  7.72it/s]


Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'sqrt', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [02:05<00:00,  7.96it/s]


Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'intra_corr', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [02:03<00:00,  8.07it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,size,False,False,t_test,True,None,None,0.001000,0.503348,True
1,sqrt,False,False,t_test,True,None,None,0.001000,0.430580,False
2,intra_corr,False,False,t_test,True,None,None,0.001000,0.486540,True


sqrt weighting is not valid here.

In [131]:
configs = [s for s in configs if not (s['weight_method']=='sqrt')]

In [132]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='ratio',
    uplift=model_uplift
)

Running A/B for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [02:09<00:00,  7.71it/s]


Running A/B for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'intra_corr', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': [], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [02:13<00:00,  7.48it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,power
0,size,False,False,t_test,True,None,None,0.001000,0.755000
1,intra_corr,False,False,t_test,True,None,None,0.001000,0.839000


intra_corr weighting provides the best statistical power.

Overall, best setup is:
- t-test
- Use stratification
- Use CUPED
- Remove 0.1% of outliers by pre-experiment features
- User intra_corr weighting of users

It gives 14% increase in statistical power comparing to default t-test without anything.